In [1]:
import pygsheets # use 'pip install pygsheets', not maintained with conda
import pandas
import datetime
import geopandas
import shapely
import pathlib
import pyogrio
import json
import numpy

# terminals

## clean up terminals

In [15]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
spreadsheet = gc.open_by_key('1tcS6Wd-Wp-LTDpLzFgJY_RSNDnbyubW3J_9HKIAys4A')

#spreadsheet = gc.open_by_key('1Rw9Xj0VIOLq94OT0zth5ub_IqCeAKYzlufdjxuezzgs') # jan 2024 dataset for EGT release
#spreadsheet = gc.open_by_key('1hOuMDpznpTKQRJ9SQGWY1ktMR_qw0vVRARrAtgeUsYQ') # Nov. 28 database for Oct 2023 GGIT release

terms_df_orig = spreadsheet.worksheet('title', 'Terminals').get_as_df(start='A3')
terms_dict_df = spreadsheet.worksheet('title', 'Data dictionary').get_as_df()
terms_acronyms_df = spreadsheet.worksheet('title', 'Acronyms').get_as_df()
terms_copyright_df = spreadsheet.worksheet('title', 'Copyright').get_as_df()

In [16]:
# remove oil export terminals
terms_df_orig = terms_df_orig.loc[terms_df_orig['Fuel']=='LNG']
# remove anything without a wiki page
terms_df_orig = terms_df_orig.loc[terms_df_orig['Wiki']!='']
# remove anything without a ComboID
terms_df_orig = terms_df_orig.loc[terms_df_orig['ComboID']!='']

## subset columns to include relevant columns

Note these columns are specified in the "Data dictionary", of the given database, and DataReleaseColumnOrder determines how they're arranged

In [ ]:
terms_dict_df_include = terms_dict_df.loc[terms_dict_df.IncludeWithDataRelease=='Yes']
terms_dict_df_include = terms_dict_df_include.sort_values('DataReleaseColumnOrder', ascending=True)
terms_dict_df_include = terms_dict_df_include[['VariableName','Definition']]

In [ ]:
no_lonlat_options = [
    'Unknown',
    'TBD',
    ''
]

In [ ]:
terms_df_subset = terms_df_orig[terms_dict_df_include['VariableName'].tolist()]

## write to Excel file

Use ExcelFormatter from Pandas to write 4 different tabs to the same file.

In [ ]:
excel_writer = pandas.ExcelWriter('GEM-GGIT-LNG-Terminals-'+str(datetime.date.today())+'.xlsx')#, engine='xlsxwriter')

terms_df_subset.to_excel(excel_writer, sheet_name='LNG Terminals '+str(datetime.date.today()), index=False)
terms_dict_df_include.to_excel(excel_writer, sheet_name='Data dictionary', index=False)
terms_acronyms_df.to_excel(excel_writer, sheet_name='Acronyms', index=False)
terms_copyright_df.to_excel(excel_writer, sheet_name='Copyright', index=False)

excel_writer.close()

# create shapely points

In [ ]:
# code to create a dataframe with WKT formatted geometry

# (1) copy, clean up
to_convert_df = terms_df_orig.copy()
to_convert_df = to_convert_df[~(to_convert_df['Latitude'].isin(no_lonlat_options)) |
                             ~(to_convert_df['Longitude'].isin(no_lonlat_options))]

# also keep the non-converted ones separate
not_converted_df = terms_df_orig.copy()
not_converted_df = not_converted_df[(not_converted_df['Longitude'].isin(no_lonlat_options)) | 
                                    (not_converted_df['Latitude'].isin(no_lonlat_options))]
# add a dummy column so that the dimensions match with converted wkt pipelines
not_converted_df.assign(ColName='geometry')
not_converted_df['geometry'] = [shapely.geometry.Point()]*not_converted_df.shape[0]
not_converted_df.reset_index(drop=True)
not_converted_gdf = geopandas.GeoDataFrame(not_converted_df, geometry=not_converted_df['geometry'])

# (2) convert all terminals
terms_df_converted = to_convert_df.copy()
terms_df_converted.assign(ColName='geometry')
terms_df_converted['geometry'] = to_convert_df[['Longitude','Latitude']].apply(shapely.geometry.Point, axis=1)
terms_df_converted = terms_df_converted.reset_index(drop=True)

# # (3) store in a GeoDataFrame, attach a projection, transform to a different one
terms_df_gdf = geopandas.GeoDataFrame(terms_df_converted, geometry=terms_df_converted['geometry'])

In [ ]:
all_terms_df = pandas.concat([terms_df_gdf, not_converted_gdf])
all_terms_df = all_terms_df.reset_index(drop=True)
all_terms_df.sort_values('ComboID', inplace=True)

In [ ]:
terms_dict_df_sorted = terms_dict_df[terms_dict_df['IncludeWithDataRelease']=='Yes'].sort_values('DataReleaseColumnOrder')
output_columns = terms_dict_df_sorted['VariableName'].tolist()

In [ ]:
all_terms_df_to_save = all_terms_df[output_columns]
all_terms_df_to_save_gdf = geopandas.GeoDataFrame(all_terms_df_to_save, geometry=all_terms_df['geometry'])

## save as GeoJSON file

In [ ]:
now_string = datetime.datetime.now().strftime('%Y-%m-%d')
filename = 'GEM-GGIT-LNG-Terminals-'+now_string+'.geojson'
all_terms_df_to_save_gdf.to_file(filename, driver='GeoJSON')
print('saved as', filename)

## save as shapefile

In [ ]:
shorter_col_names = terms_dict_df.set_index('VariableName').loc[output_columns,'10LetterName'].tolist() + ['geometry']

In [ ]:
now_string = datetime.datetime.now().strftime('%Y-%m-%d')
filename = 'GEM-GGIT-LNG-Terminals-dataset-'+now_string+'.shp'
all_terms_df_to_save_gdf.set_axis(shorter_col_names, axis=1).to_file(filename)#, driver='GeoJSON')
print('saved as', filename)
filename = 'GEM-GGIT-LNG-Terminals-dataset-'+now_string+'.shx'
all_terms_df_to_save_gdf.set_axis(shorter_col_names, axis=1).to_file(filename)
print('saved as', filename)
filename = 'GEM-GGIT-LNG-Terminals-dataset-'+now_string+'.dbf'
all_terms_df_to_save_gdf.set_axis(shorter_col_names, axis=1).to_file(filename)
print('saved as', filename)

## save as geopackage

In [ ]:
now_string = datetime.datetime.now().strftime('%Y-%m-%d')
filename = 'GEM-GGIT-LNG-Terminals-'+now_string+'.gpkg'
all_terms_df_to_save_gdf.to_file(filename, driver='GPKG', layer='terminals')
print('saved as', filename)

# pipelines

### comment out appropriate final lines to only publish oil or gas

In [54]:
gas_fuel_options = ['Gas']
gas_hydrogen_fuel_options = ['Gas',
                             'Hydrogen'
                            ]
ngl_fuel_options = ['NGL', 
                    'NGL, oil products', 
                    'Oil, NGL', 
                    'Oil, NGL, naphtha'
                   ]
oil_fuel_options = ['Oil', 
                    #'Oil, NGL', 
                    #'Oil, NGL, naphtha'
                   ]

In [55]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
#spreadsheet = gc.open_by_key('1VnhD3K8bUn-CwTGUt-XkF42jcWAaZwasKIryEO7V1j4') # may 2024 oil release
spreadsheet = gc.open_by_key('1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek') # current version

# SPECIFY HERE WHETHER YOU WANT GAS, GAS+HYDROGEN, OR OIL+NGL
#pipes_df_orig = spreadsheet.worksheet('title','Oil/NGL pipelines').get_as_df(start='A3'); github_folder = 'liquid-pipelines'; type_to_save='Oil-NGL'; which_tracker='GOIT'
#pipes_df_orig = spreadsheet.worksheet('title','Gas pipelines').get_as_df(start='A3'); github_folder = 'gas-pipelines'; type_to_save='Gas'; which_tracker='GGIT'
type_to_save='Gas-Hydrogen'; which_tracker='GGIT'; pipes_df_orig = spreadsheet.worksheet('title','Gas pipelines').get_as_df(start='A3'); github_folder = 'gas-pipelines'; 

pipes_acronyms_df = spreadsheet.worksheet('title','Acronyms').get_as_df()

In [56]:
if type_to_save=='Gas':
    pipes_dict_df = spreadsheet.worksheet('title','Data dictionary - Gas pipelines').get_as_df()
    pipes_copyright_df = spreadsheet.worksheet('title','Copyright - GGIT').get_as_df()
    pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Fuel.isin(gas_fuel_options)]
    
elif type_to_save=='Gas-Hydrogen':
    pipes_dict_df = spreadsheet.worksheet('title','Data dictionary - Gas pipelines').get_as_df()
    pipes_copyright_df = spreadsheet.worksheet('title','Copyright - GGIT').get_as_df()
    pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Fuel.isin(gas_hydrogen_fuel_options)]
    
elif type_to_save=='Oil-NGL':
    pipes_dict_df = spreadsheet.worksheet('title','Data dictionary - Oil/NGL pipelines').get_as_df()
    pipes_copyright_df = spreadsheet.worksheet('title','Copyright - GOIT').get_as_df()
    pipes_copyright_df = pandas.DataFrame(pipes_copyright_df.iloc[:,0])
    pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Fuel.isin(oil_fuel_options+ngl_fuel_options)]

    # change all oil/NGL fuels to JUST SAY Oil or NGL (rather than how we track, with different options)
    pipes_df_orig.loc[pipes_df_orig.Fuel.isin(oil_fuel_options),'Fuel'] = 'Oil'
    pipes_df_orig.loc[pipes_df_orig.Fuel.isin(ngl_fuel_options),'Fuel'] = 'NGL'

    # remove gathering, distribution lines
    pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.PipelineType.isin(['','transmission'])]

## clean up pipelines

In [57]:
# pipelines cleanup
# remove all N/A status, all empty rows (gauged by those without a Wiki page)
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Status!='N/A']
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.PipelineName!='']
#pipes_df_orig = pipes_df_orig[pipes_df_orig['Wiki']!='']

## ONLY save pipelines that have something in RouteAccuracy
this is the one column that decides whether we include it

In [58]:
# pipelines cleanup
# if the RouteAccuracy is blank, that means we don't want it included in the database
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.RouteAccuracy!='']
all_pipes_df = pipes_df_orig.copy()

# get github routes

In [59]:
# geojson_path = '/Users/baird/Dropbox/_git_ALL/_github-repos-gem/'+\
# 'GOIT-GGIT-pipeline-routes/data/individual-routes/'+github_folder+'/'
geojson_path = '/Users/baird/Google Drive/Shared drives/GEM Shared Drive/Programs/'+\
'Oil & Gas Program/Global Oil and Gas Infrastructure Trackers (GOIT, GGIT)/'+\
'GitHub repositories/GOIT-GGIT-pipeline-routes/data/individual-routes/'+github_folder+'/'

geojson_route_files = list(
    pathlib.Path(
        geojson_path
    ).rglob('?????.geojson')
)

In [60]:
geojson_projectids = [str(i).split('/')[-1][:5] for i in geojson_route_files]

In [61]:
set(all_pipes_df.ProjectID.tolist())-set(geojson_projectids)

{'P6636',
 'P6637',
 'P6638',
 'P6640',
 'P6641',
 'P6642',
 'P6644',
 'P6706',
 'P6738',
 'P6739',
 'P6740',
 'P6741',
 'P6742',
 'P6743',
 'P6744',
 'P6745',
 'P6746',
 'P6747',
 'P6748',
 'P6749',
 'P6750',
 'P6751',
 'P6752',
 'P6753',
 'P6754',
 'P6755',
 'P6756',
 'P6757',
 'P6758',
 'P6759',
 'P6760',
 'P6761',
 'P6762',
 'P6763',
 'P6764',
 'P6765',
 'P6766',
 'P6767',
 'P6768',
 'P6769',
 'P6770',
 'P6771',
 'P6772',
 'P6773',
 'P6774',
 'P6775',
 'P6776',
 'P6777',
 'P6778',
 'P6779',
 'P6780',
 'P6781',
 'P6782',
 'P6783',
 'P6784',
 'P6785',
 'P6786',
 'P6788',
 'P6790',
 'P6791',
 'P6792',
 'P6793',
 'P6794',
 'P6795',
 'P6796',
 'P6797',
 'P6798',
 'P6799',
 'P6800',
 'P6801',
 'P6802',
 'P6803',
 'P6808',
 'P6809',
 'P6810',
 'P6811',
 'P6812',
 'P6813',
 'P6814',
 'P6815',
 'P6816',
 'P6817',
 'P6818',
 'P6819',
 'P6820',
 'P6821',
 'P6842',
 'P6843'}

In [62]:
missing_list =list(set(all_pipes_df.ProjectID.tolist())-set(geojson_projectids))

all_pipes_df.loc[all_pipes_df.ProjectID.isin(missing_list)]['Researcher'].unique()

array(['HH', 'AL'], dtype=object)

In [63]:
[print(i) for i in all_pipes_df.loc[(all_pipes_df.ProjectID.isin(missing_list))&
                (all_pipes_df.Researcher=='AL')]['ProjectID'].tolist()]

P6773
P6842
P6843


[None, None, None]

In [64]:
[print(i) for i in all_pipes_df.loc[(all_pipes_df.ProjectID.isin(missing_list))&
                (all_pipes_df.Researcher=='NF')]['ProjectID'].tolist()]

[]

In [65]:
all_pipes_df.loc[(all_pipes_df.ProjectID.isin(missing_list))&
                (all_pipes_df.Researcher=='NA')]['ProjectID'].tolist()

[]

In [66]:
all_pipes_df.loc[(all_pipes_df.ProjectID.isin(missing_list))&
                (all_pipes_df.Researcher=='HH')]['ProjectID'].tolist()

['P6636',
 'P6637',
 'P6638',
 'P6640',
 'P6641',
 'P6642',
 'P6644',
 'P6706',
 'P6738',
 'P6739',
 'P6740',
 'P6741',
 'P6742',
 'P6743',
 'P6744',
 'P6745',
 'P6746',
 'P6747',
 'P6748',
 'P6749',
 'P6750',
 'P6751',
 'P6752',
 'P6753',
 'P6754',
 'P6755',
 'P6756',
 'P6757',
 'P6758',
 'P6759',
 'P6760',
 'P6761',
 'P6762',
 'P6763',
 'P6764',
 'P6765',
 'P6766',
 'P6767',
 'P6768',
 'P6769',
 'P6770',
 'P6771',
 'P6772',
 'P6774',
 'P6775',
 'P6776',
 'P6777',
 'P6778',
 'P6779',
 'P6780',
 'P6781',
 'P6782',
 'P6783',
 'P6784',
 'P6785',
 'P6786',
 'P6788',
 'P6790',
 'P6791',
 'P6792',
 'P6793',
 'P6794',
 'P6795',
 'P6796',
 'P6797',
 'P6798',
 'P6799',
 'P6800',
 'P6801',
 'P6802',
 'P6803',
 'P6808',
 'P6809',
 'P6810',
 'P6811',
 'P6812',
 'P6813',
 'P6814',
 'P6815',
 'P6816',
 'P6817',
 'P6818',
 'P6819',
 'P6820',
 'P6821']

## below are the pipelines that don't fall within the status

these may have been deleted from the spreadsheet but not from the database, or they have a N/A status, or something

In [67]:
misfits_list = list(set(geojson_projectids)-set(all_pipes_df.ProjectID.tolist()))

P2041 - oil pipeline in gas tracker  
P3162 - N/A status  
P3656 - N/A status  
P3672 - N/A status

In [68]:
sorted(misfits_list)

['P2041', 'P3162', 'P3656', 'P3672']

In [69]:
#pipes_df_orig['geometry'] = numpy.nan
for idx,pid in enumerate(all_pipes_df.ProjectID.tolist()):
    try:
        file = geojson_path + pid + '.geojson'
        file_gdf = geopandas.read_file(file)
        file_geometry_dissolved = file_gdf.dissolve().geometry
        all_pipes_df.loc[all_pipes_df.ProjectID==pid,'geometry'] = file_geometry_dissolved[0]
    except:
        print(pid," geojson file either doesn't exist or has bad geometry")

P0168  geojson file either doesn't exist or has bad geometry


/Users/baird/miniconda3/envs/gem/lib/python3.11/site-packages/shapely/set_operations.py:406: RuntimeWarning: invalid value encountered in create_collection
  collections = lib.create_collection(geometries, GeometryType.GEOMETRYCOLLECTION)
/Users/baird/miniconda3/envs/gem/lib/python3.11/site-packages/shapely/set_operations.py:406: RuntimeWarning: invalid value encountered in create_collection
  collections = lib.create_collection(geometries, GeometryType.GEOMETRYCOLLECTION)
/Users/baird/miniconda3/envs/gem/lib/python3.11/site-packages/fiona/collection.py:293: FeatureWarning: Empty field name at index 0
  self._schema = self.session.get_schema()
/Users/baird/miniconda3/envs/gem/lib/python3.11/site-packages/geopandas/geodataframe.py:652: UserWarning: Empty field name at index 0
  for feature in features_lst:


P6636  geojson file either doesn't exist or has bad geometry
P6637  geojson file either doesn't exist or has bad geometry
P6638  geojson file either doesn't exist or has bad geometry
P6640  geojson file either doesn't exist or has bad geometry
P6641  geojson file either doesn't exist or has bad geometry
P6642  geojson file either doesn't exist or has bad geometry
P6644  geojson file either doesn't exist or has bad geometry


/Users/baird/miniconda3/envs/gem/lib/python3.11/site-packages/shapely/set_operations.py:406: RuntimeWarning: invalid value encountered in create_collection
  collections = lib.create_collection(geometries, GeometryType.GEOMETRYCOLLECTION)
/Users/baird/miniconda3/envs/gem/lib/python3.11/site-packages/shapely/set_operations.py:419: RuntimeWarning: invalid value encountered in unary_union
  return lib.unary_union(collections, **kwargs)


P6706  geojson file either doesn't exist or has bad geometry
P6738  geojson file either doesn't exist or has bad geometry
P6739  geojson file either doesn't exist or has bad geometry
P6740  geojson file either doesn't exist or has bad geometry
P6741  geojson file either doesn't exist or has bad geometry
P6742  geojson file either doesn't exist or has bad geometry
P6743  geojson file either doesn't exist or has bad geometry
P6744  geojson file either doesn't exist or has bad geometry
P6745  geojson file either doesn't exist or has bad geometry
P6746  geojson file either doesn't exist or has bad geometry
P6747  geojson file either doesn't exist or has bad geometry
P6748  geojson file either doesn't exist or has bad geometry
P6749  geojson file either doesn't exist or has bad geometry
P6750  geojson file either doesn't exist or has bad geometry
P6751  geojson file either doesn't exist or has bad geometry
P6752  geojson file either doesn't exist or has bad geometry
P6753  geojson file eith

In [70]:
all_pipes_gdf = geopandas.GeoDataFrame(all_pipes_df, geometry=all_pipes_df.geometry, crs='EPSG:4326')

In [71]:
all_pipes_gdf.loc[all_pipes_gdf.geometry.isnull()].shape

(89, 138)

## subset pipeline columns to include relevant columns

Note these columns are specified in the "Data dictionary" tab of the Pipelines_Current sheet; the column IncludeWithDataRelease has a Yes if the column is included, and DataReleaseColumnOrder determines how they're arranged.

In [72]:
pipes_dict_df_include = pipes_dict_df.loc[(pipes_dict_df.IncludeWithDataRelease=='Yes')]
pipes_dict_df_include = pipes_dict_df_include.sort_values('DataReleaseColumnOrder', ascending=True)
pipes_dict_df_include = pipes_dict_df_include[['VariableName','Definition']]

In [73]:
all_pipes_gdf

,PipelineNetworkGrouping,PipelineName,SegmentName,Wiki,ProjectID,Status,Status [ref],Researcher,LastUpdated,OtherEnglishNames,...,PipelineDirectionality,QCCOwner,QCCOwner [ref],ImpactedByRussiaUkraineInvasion,EGTImport,ChinesePipelineType,ChineseNetworkPrimary,ChineseNetworkSecondary,ChineseClassificationNotes,geometry
0,,Double E Pipeline Project,,https://www.gem.wiki/Double_E_Pipeline_Project,P0061,operating,,HK,2023-08-21,,...,,,,,,,,,,"LINESTRING (-103.80632 32.52306, -103.90921 32..."
1,,Acadian Gas Pipeline System,,https://www.gem.wiki/Acadian_Gas_Pipeline_System,P0143,operating,,HK,2023-08-18,,...,,,,,,,,,,"MULTILINESTRING ((-93.71592 32.21787, -93.7035..."
2,,Algonquin Gas Transmission Pipeline,Access Northeast Gas Pipeline,https://www.gem.wiki/Algonquin_Gas_Transmissio...,P0144,cancelled,,HK,2023-08-21,,...,,,,,,,,,,"MULTILINESTRING ((-70.99878 42.25057, -71.8035..."
3,,Access South Gas Pipeline,,https://www.gem.wiki/Access_South_Gas_Pipeline,P0145,operating,,HK,2023-08-21,,...,,,,,,,,,,"LINESTRING (-80.18912 39.86926, -80.68433 39.8..."
4,,Alaska Gas Pipeline,,https://www.gem.wiki/Alaska_Gas_Pipeline,P0146,cancelled,,WA,2023-09-06,,...,,,,,,,,,,"LINESTRING (-148.20737 70.26414, -150.08087 67..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4006,,Chiba Trunk Line,,https://www.gem.wiki/Chiba_Trunk_Line,P6839,operating,https://www.tokyo-gas.co.jp/en/IR/library/pdf/...,WA,2024-08-19,,...,,,,,,,,,,GEOMETRYCOLLECTION EMPTY
4007,,Saihoku Trunk Line,,https://www.gem.wiki/Saihoku_Trunk_Line,P6840,operating,https://www.tokyo-gas.co.jp/en/IR/library/pdf/...,WA,2024-08-19,,...,,,,,,,,,,GEOMETRYCOLLECTION EMPTY
4008,,Peru LNG Gas Pipeline,,https://www.gem.wiki/Peru_LNG_Gas_Pipeline,P6841,operating,https://perulng.com/en/pipeline/,GC,2024-08-21,,...,,,,,,,,,,"MULTILINESTRING ((-76.297 -13.2451, -76.26591 ..."
4009,,Louisiana Energy Access Project Pipeline,Expansion Phase 3,,P6842,operating,https://www.spglobal.com/commodityinsights/en/...,AL,2024-08-21,"LEAP, LEAP Gathering System",...,,,,,,,,,,None


In [74]:
all_pipes_gdf_subset = all_pipes_gdf[pipes_dict_df_include['VariableName'].tolist()]

In [75]:
all_pipes_gdf_subset_with_geom = all_pipes_gdf[pipes_dict_df_include['VariableName'].tolist()+['geometry']]

## write to Excel file - this doesn't include 

Use ExcelFormatter from Pandas to write 4 different tabs to the same file.

In [76]:
#pandas.io.formats.excel.ExcelFormatter.header_style = None

excel_writer = pandas.ExcelWriter('DATA-FILES/GEM-'+which_tracker+'-'+type_to_save+'-Pipelines-'+str(datetime.date.today())[:7]+'.xlsx')#, engine='xlsxwriter')

all_pipes_gdf_subset.to_excel(excel_writer, sheet_name=type_to_save+' Pipelines '+str(datetime.date.today())[:7], index=False)
pipes_dict_df_include.to_excel(excel_writer, sheet_name='Data dictionary', index=False)
pipes_acronyms_df.to_excel(excel_writer, sheet_name='Acronyms', index=False)
pipes_copyright_df.to_excel(excel_writer, sheet_name='Copyright', index=False)

#workbook = excel_writer.book
#worksheet = excel_writer.sheets['Terminals '+str(datetime.date.today())]
#format = workbook.add_format({'text_wrap': True})

excel_writer.close()

## save as GeoJSON file

In [77]:
str(datetime.date.today())[:10]

'2024-08-22'

In [78]:
filename = 'DATA-FILES/GEM-'+which_tracker+'-'+type_to_save+'-Pipelines-'+str(datetime.date.today())[:10]+'.geojson'
all_pipes_gdf_subset_with_geom.to_file(filename, driver='GeoJSON')
print('saved as', filename)

saved as DATA-FILES/GEM-GGIT-Gas-Hydrogen-Pipelines-2024-08-22.geojson


## save as shapefile

## save as geopackage

In [ ]:
now_string = datetime.datetime.now().strftime('%Y-%m-%d')
filename = 'DATA-FILES/GEM-'+which_tracker+'-'+type_to_save+'-Pipelines-'+str(datetime.date.today())[:7]+'.gpkg'
all_pipes_gdf_subset_with_geom.to_file(filename, driver='GPKG', layer='pipelines')
print('saved as', filename)

# pull out specific pipelines